# Forecasting with Prophet - Exercise

In this exercise you will use **Facebook Prophet** to forecast the daily number of visits to the Wikipedia page of the **R programming language**.

**Why this dataset?**
The dataset contains the *log* of the daily page views of the Wikipedia article on R (2008 - 2015). It comes from the official Prophet examples and is a nice contrast with the Peyton Manning case from class:
- **Trend**: R became hugely popular between 2008 and 2015, but growth flattens towards the end. Getting the *trend flexibility* right will turn out to be crucial.
- **Weekly seasonality**: R is a tool for work and study, so people visit the page far more on weekdays than in the weekend.
- **Yearly seasonality / holidays**: dips around Christmas, New Year and the summer holidays, peaks at the start of the academic year.

**Dataset source:** `https://raw.githubusercontent.com/facebook/prophet/main/examples/example_wp_log_R.csv` - a local copy is stored in `datasets/example_wp_log_R.csv`, so the notebook also works offline.

**Goal:** build Prophet models step by step (trend → + yearly → + weekly → + holidays → tuned trend flexibility), evaluate them on a time-based hold-out test set (MAE, RMSE, MAPE) and validate them with Prophet's built-in cross-validation.

---
---
---
## Complete the tasks in bold

## 0 - Setup

**TASK: Run the cell below to import the libraries you will need.**

In [ ]:
#%pip install prophet

import logging
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# Silence the (very verbose) Stan backend messages and library FutureWarnings
logging.getLogger('cmdstanpy').disabled = True
logging.getLogger('prophet').setLevel(logging.WARNING)
warnings.filterwarnings('ignore', category=FutureWarning)

## 1 - Data Loading and Exploration

**TASK: Load `datasets/example_wp_log_R.csv` into a dataframe called `df`. Show its shape, the first rows and the `info()`. Which columns are there and what is their data type? *(Reference: 2863 rows, 2 columns.)***

### Exploratory Data Analysis

**TASK: Convert `ds` to a datetime and sort the dataframe chronologically (reset the index). Then print the date range, the number of observations, duplicated dates, missing values and the number of **missing dates** (gaps) in the daily series. Finally show the summary statistics of `y`. *(Reference: 2008-01-01 to 2015-12-31, 59 missing dates.)***

**QUESTION: Is the series complete? Do the missing dates pose a problem for Prophet? Why (not)? *(Hint: think about how Prophet models time - compared to e.g. ARIMA, which needs regularly spaced observations.)***

*Your answer:*

**TASK: Plot the full time series together with a 30-day (centered) rolling mean, and a histogram of `y`. Also show the mean of `y` per year.**

**QUESTION: Describe the trend. Is the growth constant over the whole period? What do you expect will happen if a model simply extrapolates the average growth of 2008-2014 into 2015? Do you see any outliers?**

*Your answer:*

**TASK: Explore the seasonality. First remove the trend: for the weekly pattern subtract a 30-day (centered) rolling mean from `y`; for the yearly pattern subtract a 365-day (centered) rolling mean *(why would a 30-day window not work for the yearly pattern?)*. Then visualise the detrended values per **day of the week** (e.g. a boxplot) and the mean detrended value per **month** (e.g. a bar chart). *(Reference: Saturdays and Sundays are roughly 0.3 below average; December is about 0.12 below average.)***

**QUESTION: Which seasonal pattern is the strongest: weekly or yearly? How would you explain the weekly pattern? Which months are below/above average and why? Compare with the Peyton Manning case from class.**

*Your answer:*

### Zooming In: Weekly Seasonality and Holidays

**TASK: Zoom in on the period 15 November 2013 - 31 January 2014. Plot the daily values, highlight the weekend days and add vertical lines for Thanksgiving (2013-11-28), Christmas and New Year.**

**QUESTION: What do you see around the weekends and around the holidays? Why can a smooth yearly seasonality curve not capture these effects completely?**

*Your answer:*

## 2 - Preparing the Data

Prophet expects a dataframe with exactly two columns: `ds` (datetime) and `y` (numeric). Our data already has this format - we only had to convert `ds` to a datetime and sort it.

Note that `y` is already the **log** of the page views. Modelling the log is common for count data with strong growth: multiplicative effects (e.g. "weekends have 40% fewer visits") become additive, which suits Prophet's default additive model.

### Train/Test Split

**TASK: Split the data by time: use the **last year** (everything after `df['ds'].max() - 1 year`) as test set and all earlier data as training set. Store the cutoff date in `cutoff_date`. Do **not** shuffle. Print the date ranges and sizes and plot both sets. *(Reference: 2500 training days, 363 test days.)***

**QUESTION: Why do we split by time instead of using a random split (`train_test_split` with shuffling)? Link your answer to *data leakage* (week 6).**

*Your answer:*

### Holidays

Prophet has built-in holiday calendars for many countries via `model.add_country_holidays(country_name='US')`. Since a large part of the Wikipedia (English) audience is in the US, we will use the US holidays (New Year, Memorial Day, Independence Day, Labor Day, Thanksgiving, Christmas, ...). Prophet estimates a separate effect for each holiday.

### Helper functions

**TASK: We will fit several models. To avoid repeating code, write two helper functions:
- `evaluate(y_true, y_pred)` that returns the **MAE**, **RMSE** and **MAPE** *(hint: `mean_absolute_error`, `mean_squared_error`, `mean_absolute_percentage_error` from `sklearn.metrics`)*;
- `plot_forecast(forecast, title)` that plots the training data, the test data, the forecast (`yhat`) with its uncertainty interval (`yhat_lower`, `yhat_upper`) and a vertical line at the cutoff date. Zooming in from 2013 onwards makes the plot easier to read.**

## 3 - Model Building: Iterative Approach

We build the model step by step, adding one component at a time, so that we can see what each component contributes. All models are trained on `train` and evaluated on `test`.

### Model 1: Trend Only

**TASK: Build `model1`: a Prophet model with **only a trend** (switch off yearly, weekly and daily seasonality). Fit it on the training set, predict the test period, compute MAE/RMSE/MAPE (store them in `mae1`, `rmse1`, `mape1`) and plot the forecast. *(Reference: MAE ≈ 0.289.)***

**QUESTION: What does the trend-only forecast look like? Which patterns does it miss? Does the trend itself look right for 2015?**

*Your answer:*

### Model 2: Trend + Yearly Seasonality

**TASK: Build `model2`: trend + **yearly** seasonality (weekly and daily off). Evaluate and plot it like model 1. *(Reference: MAE ≈ 0.295.)***

**QUESTION: Does adding yearly seasonality improve the forecast? Why (not)? Compare with the Peyton Manning case from class.**

*Your answer:*

### Model 3: Trend + Yearly + Weekly Seasonality

**TASK: Build `model3`: trend + yearly + **weekly** seasonality. Evaluate and plot it. *(Reference: MAE ≈ 0.266.)***

**QUESTION: How much does weekly seasonality improve the errors? Does that match what you found in the EDA?**

*Your answer:*

### Model 4: Trend + Seasonality + Holidays

**TASK: Build `model4`: like model 3, but add the **US holidays** with `add_country_holidays(country_name='US')` (call this *before* fitting). Evaluate and plot. Print `model4.train_holiday_names` to see which holidays were added. *(Reference: MAE ≈ 0.264.)***

**TASK: Plot the components of model 4 with `model4.plot_components(forecast4)`.**

**QUESTION: How much do the holidays help on the test set and why is the effect small? Look at the component plot: describe each component. What is now the largest remaining source of error?**

*Your answer:*

## 4 - Model Comparison

**TASK: Put the test metrics of models 1-4 in a dataframe `comparison` (columns `Model`, `MAE`, `RMSE`, `MAPE`) and visualise them with bar charts.**

**QUESTION: Which component contributes most? The MAPE values look very small (~3.5%). Is that a fair picture of the forecast quality? *(Hint: on which scale is `y`?)***

*Your answer:*

## 5 - Cross-Validation

A single train/test split gives us only **one** estimate, based on one (possibly unusual) year. Prophet's `cross_validation` gives a more robust estimate with a **rolling forecast origin**: it repeatedly fits the model on all data up to a cutoff date and forecasts the following `horizon`.

- `initial`: length of the first training period
- `period`: spacing between cutoff dates
- `horizon`: how far ahead we forecast

To avoid leakage we run cross-validation **on the training set only** - the test year is kept aside for the final check.

**TASK: Run `cross_validation` on `model4` (which was fitted on the training set) with `initial='1095 days'`, `period='180 days'` and `horizon='365 days'`. How many cutoffs (folds) are there? Then compute the overall metrics with `performance_metrics(..., rolling_window=1)`. *(Reference: 7 cutoffs, MAE ≈ 0.258.)***

**TASK: Plot the cross-validated MAE as a function of the forecast horizon with `plot_cross_validation_metric`.**

**QUESTION: How does the cross-validated MAE compare with the test-set MAE of model 4? How does the error evolve with the forecast horizon, and why?**

*Your answer:*

### Tuning the trend flexibility

The trend is our biggest problem. Its flexibility is controlled by `changepoint_prior_scale` (default 0.05): a higher value lets the trend bend more (risk of overfitting), a lower value makes it stiffer.

We **tune** this hyperparameter with cross-validation on the training set - *not* on the test set, because then the test set would no longer be an unbiased estimate (leakage!).

**TASK: Tune `changepoint_prior_scale` over the values `[0.001, 0.01, 0.05, 0.5]`. For each value, build the model-4 configuration (yearly + weekly + US holidays) with that value, fit it on the training set, run the same cross-validation and store the overall MAE, RMSE, MAPE and coverage in a dataframe `tuning`. Store the value with the lowest CV MAE in `best_cps`. *(This takes about 20-30 seconds. Reference: best value 0.01 with CV MAE ≈ 0.202.)***

**QUESTION: Which value of `changepoint_prior_scale` is best and why does a stiffer trend help for this series? Why do we tune with cross-validation on the training set and not by checking the test-set error for each value?**

*Your answer:*

### Model 5: Tuned Model

**TASK: Build `model5`: the model-4 configuration with `changepoint_prior_scale=best_cps`. Fit it on the training set and evaluate it **once** on the test set. Plot the forecast. *(Reference: MAE ≈ 0.161.)***

## 6 - Final Comparison

**TASK: Make a final comparison table with the test MAE, RMSE and MAPE of all 5 models, and add the cross-validated MAE for models 4 and 5. Visualise the test MAE in a bar chart.**

**QUESTION: Which model performs best? Does the improvement found with cross-validation also show up on the test set? Which two modelling decisions had the most impact?**

*Your answer:*

## 7 - Forecasting the Future

**TASK: Now that we have chosen the model, refit it on **all data** (`df`) and forecast the next 180 days. Use `make_future_dataframe(periods=180)` and plot the result with the model's `plot` method.**

**QUESTION: Why do we refit the final model on all data (including the test year) before forecasting the future? Is that data leakage? Explain.**

*Your answer:*

## Conclusion

### Summary

**QUESTION: Summarise what you learned in this exercise. Consider the EDA, the role of each Prophet component, the importance of the trend flexibility and the difference between the test-set evaluation and cross-validation.**

*Your answer:*

## Great work!

----